In [54]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.cluster import MiniBatchKMeans
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.over_sampling import SMOTE
from sklearn.neighbors import NearestNeighbors
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.ensemble import IsolationForest
from imblearn.ensemble import BalancedBaggingClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score, average_precision_score
)

In [2]:
df = pd.read_csv(r"C:\Users\ankit\OneDrive\Desktop\Research_Papers\UPI_2024_data.zip")
df.info()                     
df.describe()                 
df.nunique()                   
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 17 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   transaction id      250000 non-null  object
 1   timestamp           250000 non-null  object
 2   transaction type    250000 non-null  object
 3   merchant_category   250000 non-null  object
 4   amount (INR)        250000 non-null  int64 
 5   transaction_status  250000 non-null  object
 6   sender_age_group    250000 non-null  object
 7   receiver_age_group  250000 non-null  object
 8   sender_state        250000 non-null  object
 9   sender_bank         250000 non-null  object
 10  receiver_bank       250000 non-null  object
 11  device_type         250000 non-null  object
 12  network_type        250000 non-null  object
 13  fraud_flag          250000 non-null  int64 
 14  hour_of_day         250000 non-null  int64 
 15  day_of_week         250000 non-null  object
 16  is

transaction id        0
timestamp             0
transaction type      0
merchant_category     0
amount (INR)          0
transaction_status    0
sender_age_group      0
receiver_age_group    0
sender_state          0
sender_bank           0
receiver_bank         0
device_type           0
network_type          0
fraud_flag            0
hour_of_day           0
day_of_week           0
is_weekend            0
dtype: int64

In [3]:
df['transaction_status'] = df['transaction_status'].map({'SUCCESS': 1, 'FAILED': 0})

age_order = ['18-25', '26-35', '36-45', '46-55', '56+']
df['sender_age_group'] = pd.Categorical(df['sender_age_group'], categories=age_order, ordered=True).codes
df['receiver_age_group'] = pd.Categorical(df['receiver_age_group'], categories=age_order, ordered=True).codes

df['hour_sin'] = np.sin(2*np.pi*df['hour_of_day']/24)
df['hour_cos'] = np.cos(2*np.pi*df['hour_of_day']/24)

day_map = {'Monday':0, 'Tuesday':1, 'Wednesday':2, 'Thursday':3, 'Friday':4, 'Saturday':5, 'Sunday':6}
df['day_num'] = df['day_of_week'].map(day_map)
df['day_sin'] = np.sin(2*np.pi*df['day_num']/7)
df['day_cos'] = np.cos(2*np.pi*df['day_num']/7)

df['timestamp'] = pd.to_datetime(df['timestamp'])
df['month'] = df['timestamp'].dt.month
df['day'] = df['timestamp'].dt.day
df['year'] = df['timestamp'].dt.year
df['quarter'] = df['timestamp'].dt.quarter
df['week_of_year'] = df['timestamp'].dt.isocalendar().week.astype(int)

df['month_sin'] = np.sin(2*np.pi*df['month']/12)
df['month_cos'] = np.cos(2*np.pi*df['month']/12)

nominal_cols = ['transaction type', 'merchant_category', 'sender_state', 'sender_bank', 'receiver_bank', 'device_type', 'network_type']
df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

df = df.drop(columns=['transaction id', 'timestamp', 'day_of_week'])

bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print(df.shape)
print(df.select_dtypes(include='object').columns)
df.dtypes

(250000, 59)
Index([], dtype='object')


amount (INR)                         int64
transaction_status                   int64
sender_age_group                      int8
receiver_age_group                    int8
fraud_flag                           int64
hour_of_day                          int64
is_weekend                           int64
hour_sin                           float64
hour_cos                           float64
day_num                              int64
day_sin                            float64
day_cos                            float64
month                                int32
day                                  int32
year                                 int32
quarter                              int32
week_of_year                         int64
month_sin                          float64
month_cos                          float64
transaction type_P2M                 int64
transaction type_P2P                 int64
transaction type_Recharge            int64
merchant_category_Entertainment      int64
merchant_ca

In [31]:
X = df.drop(columns=['fraud_flag'])
y = df['fraud_flag']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train set fraud cases:", y_train.sum(), "/", len(y_train))
print("Test set fraud cases:", y_test.sum(), "/", len(y_test), "\n")

Train set fraud cases: 384 / 200000
Test set fraud cases: 96 / 50000 



In [32]:
# bsmote = BorderlineSMOTE(k_neighbors=3, random_state=42)
# X_train_res, y_train_res = bsmote.fit_resample(X_train, y_train)

# print("After BorderLine SMOTE, training class balance:")
# print(y_train_res.value_counts(), "\n")

# smote = SMOTE(random_state=28)
# X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# print("After SMOTE, training class balance:")
# print(y_train_res.value_counts(), "\n")


In [35]:
# clf = RandomForestClassifier(
#     n_estimators=150,   
#     max_depth=2,         
#     random_state=42,
#     class_weight='balanced',
#     n_jobs=-1              
# )
# clf = DecisionTreeClassifier(random_state=42, max_depth=4)
# clf.fit(X_train_res, y_train_res)

# clf = xgb.XGBClassifier(
#     random_state=42, 
#     max_depth=8,    
#     n_estimators=520,   
#     eval_metric='logloss' 
# )
# clf = xgb.XGBClassifier(
#     random_state=42, 
#     max_depth=2,                
#     n_estimators=100,
#     scale_pos_weight=520,    
#     eval_metric='logloss'
# )
clf = xgb.XGBClassifier(
    random_state=42, 
    max_depth=2,          
    min_child_weight=5,      
    scale_pos_weight=520, 
    n_estimators=150,        
    learning_rate=0.05,     
    eval_metric='logloss'
)
clf.fit(X_train_res, y_train_res)
y_pred = clf.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred, digits=3))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0      0.999     0.086     0.158     49904
           1      0.002     0.958     0.004        96

    accuracy                          0.087     50000
   macro avg      0.501     0.522     0.081     50000
weighted avg      0.997     0.087     0.157     50000

Confusion Matrix:
[[ 4268 45636]
 [    4    92]]


In [42]:
clf_brf = BalancedRandomForestClassifier(
    n_estimators=150,
    max_depth=2,          
    sampling_strategy='all', 
    replacement=True,  
    random_state=42,
    n_jobs=-1            
)
clf_brf.fit(X_train, y_train)
y_pred = clf_brf.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred, digits=3))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0      0.998     0.617     0.763     49904
           1      0.002     0.302     0.003        96

    accuracy                          0.616     50000
   macro avg      0.500     0.460     0.383     50000
weighted avg      0.996     0.616     0.761     50000

Confusion Matrix:
[[30791 19113]
 [   67    29]]


In [47]:
# X_train_normal = X_train_res[y_train_res == 0]

clf_if = IsolationForest(
    n_estimators=200,
    max_samples=256,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)
clf_if.fit(X_train_res)
raw_predictions = clf_if.predict(X_test)
y_pred_if = np.where(raw_predictions == 1, 0, 1)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_if))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_if))

Confusion Matrix:
[[44792  5112]
 [   86    10]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.90      0.95     49904
           1       0.00      0.10      0.00        96

    accuracy                           0.90     50000
   macro avg       0.50      0.50      0.47     50000
weighted avg       1.00      0.90      0.94     50000



In [53]:
clf_bbc = BalancedBaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=2), # Stick to your working shallow depth
    sampling_strategy='auto',
    replacement=False,
    random_state=42,
    n_jobs=-1
)

clf_bbc.fit(X_train_res, y_train_res)
y_pred_bbc = clf_bbc.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred, digits=3))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0      0.998     0.617     0.763     49904
           1      0.002     0.302     0.003        96

    accuracy                          0.616     50000
   macro avg      0.500     0.460     0.383     50000
weighted avg      0.996     0.616     0.761     50000

Confusion Matrix:
[[30791 19113]
 [   67    29]]


In [70]:
results = []

def evaluate_model(name, clf, X_train, y_train, X_test, y_test, results):
    if isinstance(clf, IsolationForest):
        clf.fit(X_train)
        raw_pred = clf.predict(X_test)
        y_pred = np.where(raw_pred == -1, 1, 0)       
        scores = -clf.decision_function(X_test)            
        probs = (scores - scores.min()) / (scores.max() - scores.min())  # normalize to [0,1]
    else:
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        probs = clf.predict_proba(X_test)[:, 1] if hasattr(clf, "predict_proba") else None

    pr_auc = average_precision_score(y_test, probs) if probs is not None else np.nan

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (fraud)": precision_score(y_test, y_pred, zero_division=0),
        "Recall (fraud)": recall_score(y_test, y_pred, zero_division=0),
        "F1 (fraud)": f1_score(y_test, y_pred, zero_division=0),
        "PR-AUC": pr_auc
    })

evaluate_model("Decision Tree", 
                DecisionTreeClassifier(
                max_depth=8, 
                random_state=42),
                X_train_res, y_train_res, X_test, y_test, results)

evaluate_model("Random Forest", 
                RandomForestClassifier(
                n_estimators=300,
                max_depth=8, 
                class_weight="balanced", 
                random_state=42, 
                n_jobs=-1),
                X_train_res, y_train_res, X_test, y_test, results)

evaluate_model("Balanced Bagging", 
                BalancedBaggingClassifier(
                estimator=DecisionTreeClassifier(max_depth=2), 
                random_state=42, 
                n_jobs=-1),
                X_train, y_train, X_test, y_test, results)  

evaluate_model("XGBoost", 
                xgb.XGBClassifier(
                random_state=42, 
                max_depth=2, 
                min_child_weight=5, 
                scale_pos_weight=520,
                n_estimators=150, 
                learning_rate=0.05, 
                eval_metric="logloss"),
                X_train_res, y_train_res, X_test, y_test, results)

evaluate_model("Isolation Forest",
               IsolationForest(
                n_estimators=200,
                max_samples=256,
                contamination=0.05,
                random_state=42,
                n_jobs=-1),
                X_train, y_train, X_test, y_test, results)

evaluate_model("Balanced RF",
               BalancedRandomForestClassifier(
                n_estimators=150,
                max_depth=2,          
                sampling_strategy='all', 
                replacement=True,  
                random_state=42,
                n_jobs=-1),            
                X_train, y_train, X_test, y_test, results)

In [71]:
results_df = pd.DataFrame(results).sort_values("F1 (fraud)", ascending=False).reset_index(drop=True)
print(results_df.round(4).to_string(index=False))

           Model  Accuracy  Precision (fraud)  Recall (fraud)  F1 (fraud)  PR-AUC
Isolation Forest    0.9476             0.0024          0.0625      0.0046  0.0021
         XGBoost    0.0872             0.0020          0.9583      0.0040  0.0020
Balanced Bagging    0.5579             0.0018          0.4167      0.0036  0.0017
     Balanced RF    0.6164             0.0015          0.3021      0.0030  0.0017
   Decision Tree    0.9343             0.0013          0.0417      0.0024  0.0019
   Random Forest    0.9829             0.0013          0.0104      0.0023  0.0019
